In [ ]:
# B3
import os
import torch
import pandas as pd
import librosa
import evaluate
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel


device = "cuda" if torch.cuda.is_available() else "cpu"

# load trained model
processor = WhisperProcessor.from_pretrained("./whisper-az-lora-final")
base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

# combine base and adapter
model = PeftModel.from_pretrained(base_model, "./whisper-az-lora-final")
model.to(device)

test_tsv = "../voice_dataset/az/test.tsv"
clips_dir = "../voice_dataset/az/clips/"

# take 100 random samples and adjust column name
df_test = pd.read_csv(test_tsv, sep='\t').sample(n=100, random_state=42)
df_test['text_col'] = df_test['text'] if 'text' in df_test.columns else df_test['sentence']

# build paths
df_test['audio'] = df_test['path'].apply(lambda x: os.path.join(clips_dir, x))

references = df_test['text_col'].tolist()
predictions = []

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

print(f"starting tests on {len(df_test)} audio clips...")

forced_decoder_ids = processor.get_decoder_prompt_ids(language="az", task="transcribe")

for index, row in df_test.iterrows():
    # openai whisper requires 16khz sample rate
    audio_array, _ = librosa.load(row['audio'], sr=16000)
    inputs = processor.feature_extractor(audio_array, sampling_rate=16000, return_tensors="pt").to(device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            input_features=inputs.input_features, 
            forced_decoder_ids=forced_decoder_ids,
            max_new_tokens=225)
        
    transcription = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    predictions.append(transcription)
    

print("\ncalculating Final Scores...")
final_wer = wer_metric.compute(predictions=predictions, references=references)
final_cer = cer_metric.compute(predictions=predictions, references=references)

print("-" * 30)
print("FINAL TEST RESULTS (Fine-Tuned LoRA)")
print("-" * 30)
print(f"Word Error Rate (WER): {final_wer * 100:.2f}%")
print(f"Char Error Rate (CER): {final_cer * 100:.2f}%")
print("-" * 30)

print("\nsample transcriptions:")
for i in range(3):
    print(f"Original: {references[i]}")
    print(f"Prediction: {predictions[i]}\n")